In [37]:
#!python -m pip install --upgrade pip
#%pip install pandas matplotlib seaborn scikit-learn openpyxl tensorflow xgboost aif360
#%pip install "aif360[Reductions, inFairness]"

In [38]:
import json
import pandas as pd
from pprint import pprint

from sklearn.preprocessing import LabelEncoder

from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Dense, BatchNormalization # type: ignore

from aif360.datasets import StandardDataset
from fairlearn.preprocessing import CorrelationRemover
from aif360.algorithms.inprocessing import GerryFairClassifier
from fairlearn.postprocessing import ThresholdOptimizer

random_seed = 15

In [39]:
PATH = 'C:/Users/andre/Desktop/ProjectWork_AEQUITAS_AKKODIS/'
df = (
    pd.read_excel(PATH + 'data/Dataset_Preprocessed.xlsx')
)
df.head()

,Candidate State,Age Range,Sex,Protected Category,Study Area,Study Title,Years Experience,Sector,Job Family Hiring,Job Title Hiring,...,Dynamism,Mobility,English,Residence City,Residence Province,Residence Region,Residence State,European Residence,Italian Residence,Status
0,Hired,31 - 35 years,Male,Not a protected category,Engineering,Five-year degree,[1-3],Automotive,Engineering,Consultant,...,3,2,4,TURIN,Turin,Piedmont,ITALY,Yes,Yes,Positive
1,Vivier,40 - 45 years,Female,Not a protected category,Engineering,Five-year degree,[7-10],Aeronautics,Not Specified,Not Specified,...,2,2,3,CONVERSANO,Bari,Puglia,ITALY,Yes,Yes,Negative
2,QM,36 - 40 years,Male,Not a protected category,Engineering,Five-year degree,[3-5],Consulting,Not Specified,Not Specified,...,2,2,3,CASERTA,Caserta,Campania,ITALY,Yes,Yes,Positive
3,QM,> 45 years,Male,Not a protected category,Law,Five-year degree,[7-10],Telecom,Not Specified,Not Specified,...,2,2,3,SESTO SAN GIOVANNI,Milan,Lombardy,ITALY,Yes,Yes,Positive
4,In selection,31 - 35 years,Male,Not a protected category,Engineering,Five-year degree,[3-5],Automotive,Not Specified,Not Specified,...,3,1,1,ARA,Trapani,Sicily,ITALY,Yes,Yes,Negative


In [40]:
config = {}
config['categorical_columns_custom_orders'] = {
    'Sex' : ['Female', 'Male'],
    'Status' : ['Negative', 'Positive'],
    'Italian Residence': ['No', 'Yes'],
    'Candidate State': ['Not Specified', 'Imported', 'First contact', 'In selection', 'QM', 'Vivier', 'Economic proposal', 'Hired'],
    'Age Range': ['Not Specified', '< 20 years', '20 - 25 years', '26 - 30 years', '31 - 35 years', '36 - 40 years', '40 - 45 years', '> 45 years'],
    'Years Experience': ['Not Specified', '[0]', '[0-1]', '[1-3]', '[3-5]', '[5-7]', '[7-10]', '[+10]'],
    'Study Title': ['Not Specified', 'Middle school diploma', 'High school graduation', 'Professional qualification', 'Three-year degree', 'master\'s degree', 'Five-year degree', 'Doctorate'],
    'Study Level': ['Not Specified', 'Middle school diploma', 'High school graduation', 'Professional qualification', 'Three-year degree', 'master\'s degree', 'Five-year degree', 'Doctorate'],
    'Current Ral': ['Not Specified', 'Not available', '- 20 K', '20-22 K', '22-24 K', '24-26 K', '26-28 K', '28-30 K', '30-32 K', '32-34 K', '34-36 K', '36-38 K', '38-40 K', '40-42 K', '42-44 K', '44-46 K', '46-48 K', '48-50 K', '+ 50 K'],
    'Expected Ral': ['Not Specified', 'Not available', '- 20 K', '20-22 K', '22-24 K', '24-26 K', '26-28 K', '28-30 K', '30-32 K', '32-34 K', '34-36 K', '36-38 K', '38-40 K', '40-42 K', '42-44 K', '44-46 K', '46-48 K', '48-50 K', '+ 50 K'],
    'Ral Maximum': ['Not Specified', 'Not Avail.', '- 20K', '20K', '20-22K', '22-24K', '24-26K', '26-28K', '28-30K', '30-32K', '32-34K', '34-36K', '36-38K', '38-40K', '40-42K', '42-44K', '44-46K', '46-48K', '48-50k', '+50K'],
    'Minimum Ral': ['Not Specified', 'Not Avail.', '- 20K', '20K', '20-22K', '22-24K', '24-26K', '26-28K', '28-30K', '30-32K', '32-34K', '34-36K', '36-38K', '38-40K', '40-42K', '42-44K', '44-46K', '46-48K', '48-50K', '+50K'],
    'Overall' : ['Not Specified', '~ 1 - Low', '1 - Low', '~ 2 - Medium', '2 - Medium', '~ 3 - High', '3 - High', '~ 4 - Top', '4 - Top']
}

In [41]:
columns_type = {}
for col in df.columns:
    if pd.api.types.is_string_dtype(df[col]):
        columns_type[col] = 'cat'
    elif pd.api.types.is_numeric_dtype(df[col]):
        columns_type[col] = 'num'
pprint(columns_type)

{'Age Range': 'cat',
 'Candidate State': 'cat',
 'Comunication': 'num',
 'Current Ral': 'cat',
 'Dynamism': 'num',
 'English': 'num',
 'European Residence': 'cat',
 'Event_Feedback': 'cat',
 'Expected Ral': 'cat',
 'Italian Residence': 'cat',
 'Job Family Hiring': 'cat',
 'Job Title Hiring': 'cat',
 'Maturity': 'num',
 'Minimum Ral': 'cat',
 'Mobility': 'num',
 'Overall': 'cat',
 'Protected Category': 'cat',
 'Ral Maximum': 'cat',
 'Residence City': 'cat',
 'Residence Province': 'cat',
 'Residence Region': 'cat',
 'Residence State': 'cat',
 'Sector': 'cat',
 'Sex': 'cat',
 'Status': 'cat',
 'Study Area': 'cat',
 'Study Level': 'cat',
 'Study Title': 'cat',
 'Technical Skills': 'num',
 'Years Experience': 'cat'}


In [42]:
encoding_mappings = {}
for col in [col for col, t in columns_type.items() if t == 'cat']:
    if col in config['categorical_columns_custom_orders']:
        df[col] = pd.Categorical(df[col], categories=config['categorical_columns_custom_orders'][col], ordered=True).codes
        encoding_mappings[col] = {cat: i for i, cat in enumerate(config['categorical_columns_custom_orders'][col])}
    else:
        encoder = LabelEncoder()
        df[col] = encoder.fit_transform(df[col].astype(str))
        encoding_mappings[col] = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
df.head()

,Candidate State,Age Range,Sex,Protected Category,Study Area,Study Title,Years Experience,Sector,Job Family Hiring,Job Title Hiring,...,Dynamism,Mobility,English,Residence City,Residence Province,Residence Region,Residence State,European Residence,Italian Residence,Status
0,7,4,1,0,1,6,3,1,3,5,...,3,2,4,689,102,13,14,1,1,1
1,5,6,0,0,1,6,6,0,4,8,...,2,2,3,218,8,14,14,1,1,0
2,4,5,1,0,1,6,4,2,4,8,...,2,2,3,157,22,4,14,1,1,1
3,4,7,1,0,3,6,6,13,4,8,...,2,2,3,636,57,9,14,1,1,1
4,3,4,1,0,1,6,4,1,4,8,...,3,1,1,39,98,16,14,1,1,0


## Train

In [43]:
target = 'Status'
sensitive = ['Sex', 'Italian Residence']

### Dataset Preparation

In [44]:
df = shuffle(df, random_state=random_seed)

X = df.copy()
y = df[target]
s = df[sensitive]
X_train_split, X_test_split, y_train_split, y_test_split, s_train_split, s_test_split = train_test_split(X, y, s, test_size=0.2, random_state=random_seed, stratify=y)

In [45]:
train_df = X_train_split.copy()
train_df[target] = y_train_split.values
train_df[sensitive] = s_train_split.values

train_ds = StandardDataset(
    train_df,
    label_name=target,
    favorable_classes=[1], # the value considered favorable (1)
    protected_attribute_names=sensitive,
    privileged_classes=[[1] , [1]], # values considered privileged
)

In [46]:
test_df = X_test_split.copy()
test_df[target] = y_test_split.values
test_df[sensitive] = s_test_split.values

test_ds = StandardDataset(
    test_df,
    label_name=target,
    favorable_classes=[1], # the value considered favorable (1)
    protected_attribute_names=sensitive,
    privileged_classes=[[1] , [1]], # values considered privileged
)

In [47]:
predictions = {}

### Pre-Processing

In [48]:
cr = CorrelationRemover(sensitive_feature_ids=sensitive, alpha=1)

X_train_cr = cr.fit_transform(train_df)
X_train_cr_df = pd.DataFrame(X_train_cr)

X_test_cr = cr.transform(test_df)
X_test_cr_df = pd.DataFrame(X_test_cr)

In [49]:
def create_model(seed, input_dim):
    tf.random.set_seed(seed)
    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(64, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

models = {
    'Logistic Regression': LogisticRegression(),
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'Naive Bayes': GaussianNB(),
    'XGBoost': XGBClassifier(),
    'KNN': KNeighborsClassifier(),
    'Neural Network': create_model(random_seed, X_train_cr_df.shape[1]),
}

In [50]:
for model_name, model in models.items():
    print(f"Training {model_name}...")
    if model_name == 'Neural Network':
        model.fit(X_train_cr_df, y_train_split, epochs=10, batch_size=32, verbose=0)
        predictions[f"{model_name}_preprocessed_cr"] = model.predict(X_test_cr_df).flatten()
    else:
        model.fit(X_train_cr_df, y_train_split)
        predictions[f"{model_name}_preprocessed_cr"] = model.predict(X_test_cr_df)

    if model_name in ['Linear Regression', 'XGBoost', 'Neural Network']:
        predictions[f"{model_name}_preprocessed_cr"] = (predictions[f"{model_name}_preprocessed_cr"] > 0.5).astype(int)
    
    print(f"{model_name} trained.")
    temp = predictions[f"{model_name}_preprocessed_cr"]
    print(f"Preprocessed predictions for {model_name}: {temp}")
    

Training Logistic Regression...
Logistic Regression trained.
Preprocessed predictions for Logistic Regression: [0 0 0 0 0 0 0 1 1 1 0 1 1 0 0 0 1 1 0 1 0 0 1 0 0 1 0 1 1 1 0 1 0 1 0 0 0
 0 0 0 0 0 1 0 1 0 0 0 1 0 0 0 1 1 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1
 1 0 0 1 0 1 0 1 0 1 1 0 1 0 0 0 1 0 0 0 1 1 0 0 0 0 1 1 0 0 1 1 0 0 0 1 0
 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 1 0 0 1 0 1 0 0 1 0 0 0 0 1 1 0 0 0 0 0 1 1
 1 0 1 1 0 0 0 0 0 1 1 0 0 1 0 1 0 0 0 1 1 0 1 0 0 1 0 1 0 0 0 0 0 0 0 0 1
 1 1 0 0 0 1 1 1 0 1 0 1 1 0 1 1 0 1 0 0 0 0 1 1 0 0 0 0 0 0 0 1 1 1 0 0 0
 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 1 0 1 0 0 0 0 0 0 1 1 0 0 0 0 0 1 0 0
 0 0 0 0 1 0 1 1 0 0 0 1 0 1 0 0 1 1 0 0 0 1 1 0 0 0 0 1 0 1 1 0 0 0 1 0 0
 0 0 1 1 0 1 1 1 0 0 0 0 0 0 0 1 0 0 0 0 0 1 1 0 0 1 1 0 0 0 0 0 0 0 1 1 0
 0 0 0 0 0 1 1 1 0 1 0 0 0 1 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1
 0 0 1 0 0 0 1 0 1 0 0 0 1 1 0 0 1 1 0 0 1 0 1 0 1 1 0 0 1 0 1 0 0 1 0 0 1
 0 1 0 0 0 0 0 1 1 1 1 0 1 0 0 0 1 1 0 1 1 0 1 0 0 1 0 0 1 1 1 0

c:\Users\andre\AppData\Local\Programs\Python\Python38\lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


XGBoost trained.
Preprocessed predictions for XGBoost: [0 0 0 0 0 0 0 1 1 1 0 1 1 0 0 0 1 1 0 1 0 0 1 0 0 1 0 1 1 1 0 1 0 1 0 0 0
 0 0 0 0 0 1 0 1 0 0 0 1 0 0 0 1 1 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1
 1 0 0 1 1 1 0 1 0 1 1 0 1 0 0 0 1 0 0 0 1 1 0 0 0 1 1 1 0 0 1 1 0 0 0 1 0
 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 1 0 0 1 0 1 0 0 1 0 0 0 0 1 1 0 0 0 0 0 1 1
 1 0 1 1 0 0 0 0 0 1 1 0 0 1 0 1 0 0 0 1 1 0 1 0 0 1 0 1 0 0 0 0 0 0 0 1 1
 1 1 0 0 0 1 1 1 0 1 0 1 1 0 1 1 0 1 0 0 0 0 1 1 0 0 0 0 0 0 0 1 1 1 0 0 0
 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 1 0 0 0 0 0 0 1 1 0 0 0 0 0 1 0 0
 0 0 0 0 1 0 0 1 0 0 0 1 0 1 0 0 1 1 0 1 0 1 1 0 0 0 0 1 0 1 1 0 1 0 1 0 0
 0 0 1 1 0 1 1 1 0 0 0 0 0 0 0 1 0 0 0 0 0 1 1 0 0 1 1 0 0 0 0 0 0 0 1 1 0
 0 0 0 0 0 1 1 1 0 1 0 0 0 1 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1
 0 0 1 0 0 0 1 0 1 0 0 0 1 1 0 0 1 1 0 0 1 0 1 0 1 1 0 0 1 0 1 0 0 1 0 0 1
 0 1 0 0 0 0 0 1 1 1 1 0 1 0 0 0 1 1 0 1 1 0 1 0 0 1 0 0 1 1 1 0 0 1 0 0 0
 1 0 1 1 0 1 1 0 0 0 1 0 1 1 1 0 0 0 0 0 1 1 

### In-Processing

In [51]:
models = {
    'Linear Regression': LinearRegression(),
}

In [52]:
for model_name, model in models.items():
    gfc = GerryFairClassifier(
        C=10,
        gamma=0.01,
        fairness_def='FP',
        max_iters=10,
        printflag=False,
        heatmapflag=False,
        heatmap_iter=10,
        heatmap_path='.',
        predictor=model
    )
    gfc.fit(train_ds)
    pred_gfc = gfc.predict(test_ds)
    predictions[f"{model_name}_inprocessed_gfc"] = pred_gfc.labels.ravel()
    temp = predictions[f"{model_name}_inprocessed_gfc"]
    print(f"Inprocessed predictions for {model_name}: {temp}")


Inprocessed predictions for Linear Regression: [0 0 0 0 0 0 0 0 1 1 0 1 1 0 0 0 0 0 0 1 0 0 1 0 0 1 0 1 1 0 0 1 0 1 0 0 0
 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1
 0 0 0 1 0 0 0 0 0 0 1 0 1 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0
 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1
 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1
 1 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0
 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0
 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 1 1 0 1 0 1 1 0 0 0 0 1 0 1 1 0 1 0 1 0 0
 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 1 1 0
 0 0 0 0 0 0 1 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1
 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 1 0 1 0 0 0 0 0 0
 0 1 0 0 0 0 0 0 0 1 1 0 1 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 1 1 0 0 0 0 0 0
 0 0 1 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 1 0 1 1 1 

### Post-Processing

In [53]:
def create_model(seed, input_dim):
    tf.random.set_seed(seed)
    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(64, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

models = {
    'Logistic Regression': LogisticRegression(),
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'Naive Bayes': GaussianNB(),
    'XGBoost': XGBClassifier(),
    'KNN': KNeighborsClassifier(),
    'Neural Network': create_model(random_seed, train_df.shape[1]),
}

In [54]:
for model_name, model in models.items():
    print(f"Training {model_name}...")
    if model_name == 'Neural Network':
        model.fit(train_df, y_train_split, epochs=10, batch_size=32, verbose=0)
        predictions[f"{model_name}_postprocessed_to"] = model.predict(test_df).flatten()
    else:
        model.fit(train_df, y_train_split)
        predictions[f"{model_name}_postprocessed_to"] = model.predict(test_df)

    print(f"{model_name} trained.")

Training Logistic Regression...
Logistic Regression trained.
Training Linear Regression...
Linear Regression trained.
Training Decision Tree...
Decision Tree trained.
Training Naive Bayes...
Naive Bayes trained.
Training XGBoost...


c:\Users\andre\AppData\Local\Programs\Python\Python38\lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


XGBoost trained.
Training KNN...
KNN trained.
Training Neural Network...
17/17 [==============================] - 0s 4ms/step
Neural Network trained.


In [55]:
for model_name, model in models.items():
    to = ThresholdOptimizer(
        estimator=model,
        constraints='demographic_parity',
        objective='accuracy_score',
        prefit=True,
        grid_size=1000,
        flip=False,
        predict_method='predict'
    )
    to.fit(train_df, y_train_split, sensitive_features=s_train_split)
    pred_to = to.predict(test_df, sensitive_features=s_test_split)
    predictions[f"{model_name}_postprocessed_to"] = pred_to.ravel()
    temp = predictions[f"{model_name}_postprocessed_to"]
    print(f"Postprocessed predictions for {model_name}: {temp}")

Postprocessed predictions for Logistic Regression: [0 0 0 0 0 0 0 1 1 1 0 1 1 0 0 0 1 1 0 1 0 0 1 0 0 1 0 1 1 1 0 1 0 1 0 0 0
 0 0 0 0 0 1 0 1 0 0 0 1 0 0 0 1 1 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1
 1 0 0 1 0 0 0 1 0 1 1 0 1 0 0 0 1 0 0 0 1 1 0 0 0 1 1 1 0 0 1 1 0 0 0 1 0
 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 1 0 1 0 0 1 0 0 0 0 1 1 0 0 0 0 0 1 1
 1 0 1 1 0 0 0 0 0 1 1 0 0 1 0 1 0 0 0 1 1 0 1 0 0 1 0 1 0 0 0 0 0 0 0 0 1
 1 1 0 0 0 0 0 1 0 1 0 1 1 0 1 1 0 1 0 0 0 0 1 1 0 0 0 0 0 0 0 1 1 1 0 0 0
 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 1 0 0 0 0 0 0 1 1 0 0 0 0 0 1 0 0
 0 0 0 0 1 0 0 1 0 0 0 1 0 1 0 0 1 1 0 1 0 1 1 0 0 0 0 1 0 1 1 0 1 0 1 0 0
 0 0 1 0 0 1 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 1 1 0 0 1 1 0 0 0 0 0 0 0 1 1 0
 0 0 0 0 0 1 1 1 0 0 0 0 0 1 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1
 0 0 1 0 0 0 1 0 1 0 0 0 1 1 0 0 0 1 0 0 1 0 1 0 1 1 0 0 1 0 1 0 0 1 0 0 1
 0 1 0 0 0 0 1 1 1 1 1 0 1 0 0 0 1 1 0 1 1 0 1 0 0 1 0 0 1 1 1 0 0 1 0 0 0
 1 0 1 1 0 1 1 0 0 0 1 0 1 1 1 0 0 0 0 0 1 1 0 1 

c:\Users\andre\AppData\Local\Programs\Python\Python38\lib\site-packages\fairlearn\postprocessing\_threshold_optimizer.py:309: UserWarning: The value of `prefit` is `True`, but `check_is_fitted` raised `NotFittedError` on the base estimator.

If the provided base estimator has been fitted, this could mean that (1) its implementation does not conform to the sklearn estimator API, or (2) the enclosing ThresholdOptimizer has been cloned (for instance by `sklearn.model_selection.cross_validate`).

In case (1), please file an issue with the base estimator developers, but continue to use the enclosing ThresholdOptimizer with `prefit=True`. In case (2), please use `prefit=False`.
  warn(BASE_ESTIMATOR_NOT_FITTED_WARNING.format(type(self).__name__))


 1/17 [>.............................] - ETA: 0s

c:\Users\andre\AppData\Local\Programs\Python\Python38\lib\site-packages\fairlearn\postprocessing\_interpolated_thresholder.py:107: UserWarning: The value of `prefit` is `True`, but `check_is_fitted` raised `NotFittedError` on the base estimator.

If the provided base estimator has been fitted, this could mean that (1) its implementation does not conform to the sklearn estimator API, or (2) the enclosing InterpolatedThresholder has been cloned (for instance by `sklearn.model_selection.cross_validate`).

In case (1), please file an issue with the base estimator developers, but continue to use the enclosing InterpolatedThresholder with `prefit=True`. In case (2), please use `prefit=False`.
  warn(BASE_ESTIMATOR_NOT_FITTED_WARNING.format(type(self).__name__))


17/17 [==============================] - 0s 4ms/step
Postprocessed predictions for Neural Network: [0 0 0 0 0 0 0 0 1 1 0 1 1 0 0 0 1 1 0 1 0 0 1 0 0 1 0 1 1 1 0 1 1 1 0 0 0
 0 0 0 0 0 1 0 1 0 0 0 1 0 0 0 1 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1
 1 0 0 1 1 1 0 0 0 0 1 0 1 0 0 0 1 0 0 0 1 1 0 0 0 1 1 1 0 0 1 1 0 0 0 1 0
 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 1 0 1 0 0 1 0 0 0 0 1 1 0 0 0 0 0 1 1
 1 0 1 1 0 0 0 0 0 1 1 0 0 1 0 1 0 0 0 1 1 0 1 0 0 1 0 0 0 0 0 0 0 0 0 1 1
 1 1 0 0 0 1 1 1 0 1 0 1 1 1 1 1 0 1 0 0 0 0 1 1 0 0 0 0 0 0 0 1 1 1 0 0 0
 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 1 0 0 0 0 0 0 1 1 0 0 0 0 0 1 0 0
 0 0 0 0 1 0 0 1 0 0 0 1 0 1 0 0 1 1 0 1 0 1 1 0 0 0 0 1 0 0 1 0 1 0 1 0 0
 0 0 1 1 0 1 1 1 0 1 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 1 0 0 0 0 0 0 0 1 1 0
 0 0 0 0 0 1 1 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1
 0 0 1 0 0 0 1 0 1 0 0 0 1 1 0 0 1 1 0 0 1 0 1 0 1 1 0 0 1 0 1 0 0 1 0 0 1
 0 1 0 0 0 0 0 1 1 1 1 0 1 0 0 0 1 1 0 1 1 0 1 0 0 1 0 0 1 1 1 0 0 1 0 0 0
 

## Save

In [56]:
reference = {}
for model_name, preds in predictions.items():
    reference[model_name] = y_test_split.values
serializable_data = {
  'predictions': {k: v.tolist() for k, v in predictions.items()},
  'reference':   y_test_split.values.tolist(),
  'sensitive':   s_test_split.values.tolist(),
  'sensitive_name': sensitive,
}

In [57]:
with open(PATH + 'data/predictions.json', 'w') as f:
  json.dump(serializable_data, f, indent=4)